<a href="https://colab.research.google.com/github/fbeilstein/bioinformatics/blob/master/practice_05_single_cell_microfluidics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Cellular Individuality (Single-Cell Droplet Microfluidics)**

In this notebook, we move from bulk tissue averages to the resolution of **individual cells**. Biological tissues are highly heterogeneous; a bulk RNA-seq sample averages out the transcriptional bursting of individual cells and obscures rare cellular microenvironments.

The instrumental method we are studying is **Droplet Microfluidics (e.g., 10x Genomics Chromium)**. In this process, single cells are flowed through microscopic channels and encapsulated into oil droplets alongside hydrogel beads. These beads contain millions of DNA primers covered in two important barcodes:
1.  **Cell Barcode:** Identifies which cell (droplet) the RNA came from.
2.  **Unique Molecular Identifier (UMI):** Identifies the specific, original mRNA molecule to correct for PCR amplification bias and collisions.

Our primary computational challenges involve **Sparsity Filtering & Robust Statistics**. Because most droplets are actually empty, and some cells undergo apoptosis (releasing mitochondrial RNA into the cytoplasm during cell lysis), we must dynamically filter out empty droplets, dying cells, and doublets using **Median Absolute Deviation (MAD)**.

We will use the industrial tools **Scanpy** and **AnnData** to process a raw, unfiltered 10x Genomics Peripheral Blood Mononuclear Cell (PBMC) dataset.


### **Part 1: Environment Setup**

We will utilize `scanpy`, the dominant Python framework for single-cell analysis, which heavily relies on `anndata` (Annotated Data) objects to manage highly sparse, high-dimensional matrices.


In [ ]:
import os
from IPython.display import clear_output

# Install scanpy, anndata, and standard libraries
!pip install -q scanpy anndata numpy pandas scipy matplotlib seaborn

clear_output()
print("Installations successful.")


### **Part 2: Dataset Download**

We will download the **raw, unfiltered** PBMC3k dataset from 10x Genomics. Unlike pre-filtered datasets that only include successful cells, the raw matrix includes all ~737,000 possible droplet barcodes. Most of these are "empty droplets" that only contain ambient RNA from the microfluidic soup.


In [ ]:
!wget --no-check-certificate "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_raw_gene_bc_matrices.tar.gz"

import tarfile
from IPython.display import clear_output

filename = "pbmc3k_raw_gene_bc_matrices.tar.gz"
print("Extracting dataset...")
with tarfile.open(filename, "r:gz") as tar:
    tar.extractall()

clear_output()
print("Raw data downloaded and extracted to the 'raw_gene_bc_matrices/hg19' folder.")

### **Part 3: Reading Data and the AnnData Structure**

An `AnnData` object stores:
*   `.X`: The sparse count matrix (Cells $\times$ Genes).
*   `.obs`: Observations (metadata about the cells/droplets, like total UMIs).
*   `.var`: Variables (metadata about the genes, like GC content or chromosome).

We use `scanpy.read_10x_mtx` to parse the Matrix Market (`.mtx`) files generated by the Cell Ranger pipeline

In [ ]:

import scanpy as sc
import pandas as pd
import numpy as np

# Read the raw 10x matrix
print("Loading raw 10x matrix into AnnData...")
adata = sc.read_10x_mtx(
    'raw_gene_bc_matrices/hg19/',  # the directory with the `.mtx` file
    var_names='gene_symbols',      # use gene symbols for the variable names (variables-index)
    cache=True                     # write a cache file for faster subsequent reading
)

# Ensure unique gene names
adata.var_names_make_unique()

print(f"\nRaw Matrix Shape: {adata.shape[0]:,} cells (droplets) × {adata.shape[1]:,} genes")


### **Part 4: The Knee Plot (Finding Real Cells)**

Almost 700,000 droplets are empty, capturing only floating ambient RNA released by cells that died during the tissue dissociation process (cell lysis kinetics).

We can plot a **Knee Plot** (or Barcode Rank Plot). We rank every droplet by its total UMI count and plot it on a log-log scale. The sharp dropoff (the "knee") separates the real, intact cells from the background noise of empty droplets.


In [ ]:

import matplotlib.pyplot as plt

# Calculate the total UMIs per droplet
umi_counts = np.array(adata.X.sum(axis=1)).flatten()

# Sort droplets in descending order of UMI counts
sorted_umi_counts = np.sort(umi_counts)[::-1]

# Plot the Knee Plot
plt.figure(figsize=(8, 5))
plt.plot(sorted_umi_counts, color='blue', linewidth=2)
plt.xscale('log')
plt.yscale('log')
plt.axhline(y=500, color='red', linestyle='--', label='Typical Cell Threshold (500 UMIs)')
plt.title('Barcode Rank Plot (Knee Plot)')
plt.xlabel('Barcodes (Droplets) ranked by UMI count')
plt.ylabel('Total UMI Count')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### **Part 5: Quality Control (QC) Metrics**

Even among "real" cells, we must filter out dying cells. During apoptosis, the cell membrane ruptures, leaking cytoplasmic mRNA into the environment. However, the mitochondria are larger and remain trapped inside the dying cell's skeleton.

Therefore, a dying cell will appear to have an unusually high percentage of **Mitochondrial RNA**. We calculate this percentage for every cell.


In [ ]:
# Annotate the group of mitochondrial genes as 'mt'
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# sc.pp.calculate_qc_metrics calculates metrics like total counts and pct_counts_mt
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# Let's filter out the empty droplets first (keeping droplets with > 500 UMIs)
adata = adata[adata.obs.total_counts > 500, :].copy()

print(f"Cells remaining after basic threshold: {adata.n_obs:,}")

# Plot distributions of QC metrics
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, size=1)


### **Part 6: Robust Statistics (Median Absolute Deviation)**

Instead of arbitrarily deciding that "any cell with > 5% mitochondrial RNA is dead", modern pipelines use **Median Absolute Deviation (MAD)**.

MAD is a robust measure of statistical dispersion. Unlike Standard Deviation ($\sigma$), MAD is not heavily skewed by extreme outliers.

$$ \text{MAD} = \text{median}(|X_i - \text{median}(X)|) $$

A common dynamic filter is to drop any cell whose metric is more than $3 \times \text{MAD}$ away from the median.


In [ ]:

# Let's calculate MAD for the mitochondrial percentage
pct_mt = adata.obs['pct_counts_mt'].values
median_mt = np.median(pct_mt)
mad_mt = np.median(np.abs(pct_mt - median_mt))

# To scale MAD to approximate standard deviation for a normal distribution, multiply by 1.4826
mad_scaled = mad_mt * 1.4826

threshold = median_mt + (3 * mad_scaled)

print(f"Median % MT: {median_mt:.2f}%")
print(f"MAD Scaled:  {mad_scaled:.2f}%")
print(f"Dynamic Outlier Threshold (Median + 3 MADs): {threshold:.2f}%")

plt.figure(figsize=(8, 4))
plt.hist(pct_mt, bins=50, color='gray', edgecolor='black')
plt.axvline(threshold, color='red', linestyle='dashed', linewidth=2, label=f'3 MADs Threshold ({threshold:.1f}%)')
plt.title("Distribution of Mitochondrial RNA %")
plt.xlabel("% Mitochondrial RNA")
plt.ylabel("Number of Cells")
plt.legend()
plt.show()


### **Homework: Single-Cell Microfluidics QC**

Use the concepts and tools demonstrated above to analyze and process the `adata` object.


#### **Task 1: Filter by MAD**

Using the `threshold` calculated in Part 6, filter the `adata` object to remove dying cells.
Also, filter out "doublets" (droplets that accidentally captured two cells) by dropping cells that have more than 2,500 genes expressed (`n_genes_by_counts`).
Save this clean matrix into a new variable `adata_clean`.


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 2: Implement MAD Function**

Write a pure Python/NumPy function `compute_mad(data_array)` that calculates and returns the scaled MAD for any given 1D numerical array.
*Don't forget to multiply the final median by the scale factor 1.4826.*

In [ ]:
def compute_mad(data_array):
  """
  Returns the scaled Median Absolute Deviation of a 1D array.
  """
  # ---- YOUR CODE HERE ----
  return 0.0


#### **Task 3: Highest Expression**

Which gene is the most highly expressed across the entire `adata_clean` dataset?
Find the gene name (from `adata_clean.var_names`) that has the highest sum of raw counts across all valid cells.

In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 4: UMI Collision Math**

A typical 10x Genomics UMI barcode is 10 nucleotides long.
1. How many unique UMIs are possible?
2. If a specific mRNA transcript is highly expressed and captured 50,000 times in a single droplet, what is the probability of a UMI collision?
(Use the expected number of distinct UMIs captured: $E = N \times (1 - (1 - \frac{1}{N})^k)$, where $N$ is total possible UMIs and $k$ is the number of captured molecules). Calculate how many *fewer* UMIs you observe compared to the true $k$.


In [ ]:
# ---- YOUR CODE HERE ----

#### **Task 5: Interpretation Questions**

Answer the following in a text cell:

1. Why do we see a sharp "knee" in the barcode rank plot rather than a smooth, continuous curve?
2. Why is standard deviation ($\sigma$) a poor metric for outlier detection in raw single-cell data compared to MAD?
3. What biological event causes a cell to have 50% mitochondrial RNA in a droplet microfluidics experiment?
4. If a droplet contains two cells (a doublet), how will it appear in our Quality Control metrics (`total_counts` and `n_genes_by_counts`)?
5. Why are Unique Molecular Identifiers (UMIs) attached *before* PCR amplification during library preparation?